# 05 Dual Vector Indexes

**Project:** Enterprise Multi-Document RAG Assistant

**Notebook:** `05-dual-vector-indexes.ipynb`

In [1]:
# !pip install numpy==1.26.4
# !pip install faiss-cpu==1.8.0

In [2]:
# =====================================================
# Notebook 05
# Dual Vector Indexes
# =====================================================

import os
import pickle

import pandas as pd
import numpy as np

import faiss

from sentence_transformers import SentenceTransformer

d:\Books\projects-nlp-transformers-learning\.projectnlps\Lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


In [3]:
children_df = pd.read_csv("artifacts/child_chunks.csv")


children_df.head()

,child_id,parent_id,document_id,content
0,5f2b6fb4-0d03-4817-900a-945162a178c2,7d09d047-2858-43c4-bf8b-d40519129087,1fe99400-06e1-4964-956b-3861687e7da4,AI Platform Roadmap\nSearch Infrastructure\nRe...
1,047bf5df-d450-4bbe-92c2-2396ed3a7b30,7d09d047-2858-43c4-bf8b-d40519129087,1fe99400-06e1-4964-956b-3861687e7da4,ructure rollout.\nSearch latency improved sign...
2,b400d323-ca7c-4813-ac4e-a2ba69786f3f,7d09d047-2858-43c4-bf8b-d40519129087,1fe99400-06e1-4964-956b-3861687e7da4,eval implementation.\nSemantic search quality ...
3,e68fd4e0-8796-416e-b24b-46ccae0f8d1d,a1c5bb38-4de7-4ca4-805a-7e3fc0419867,1fe99400-06e1-4964-956b-3861687e7da4,er deployment planned.\nEnterprise RAG platfor...
4,e001567a-7134-4290-9ebb-e7e07e069dcb,a1c5bb38-4de7-4ca4-805a-7e3fc0419867,1fe99400-06e1-4964-956b-3861687e7da4,ent | Status\nVector Search | Production\nHybr...


In [4]:
metadata_df = pd.read_csv("artifacts/document_metadata.csv")

metadata_df.head()

,document_id,filename,department,doc_type,created_at,content,char_count
0,1fe99400-06e1-4964-956b-3861687e7da4,AI_Roadmap.txt,engineering,corporate_document,2026-01-01,\n# AI Platform Roadmap\n\n## Search Infrastru...,516
1,b638264b-b788-442e-a09d-19176466fba8,Annual_Report_2026.txt,finance,corporate_document,2026-01-01,\n# Annual Financial Report 2026\n\n## Revenue...,479
2,31516a79-2cf5-4080-ac66-66b338d6ebff,Benefits_2026.txt,hr,corporate_document,2026-01-01,\n# Employee Benefits 2026\n\n## Health Insura...,610


In [5]:
children_df = children_df.merge(
    metadata_df[["document_id", "department"]], on="document_id", how="left"
)

children_df.head()

,child_id,parent_id,document_id,content,department
0,5f2b6fb4-0d03-4817-900a-945162a178c2,7d09d047-2858-43c4-bf8b-d40519129087,1fe99400-06e1-4964-956b-3861687e7da4,AI Platform Roadmap\nSearch Infrastructure\nRe...,engineering
1,047bf5df-d450-4bbe-92c2-2396ed3a7b30,7d09d047-2858-43c4-bf8b-d40519129087,1fe99400-06e1-4964-956b-3861687e7da4,ructure rollout.\nSearch latency improved sign...,engineering
2,b400d323-ca7c-4813-ac4e-a2ba69786f3f,7d09d047-2858-43c4-bf8b-d40519129087,1fe99400-06e1-4964-956b-3861687e7da4,eval implementation.\nSemantic search quality ...,engineering
3,e68fd4e0-8796-416e-b24b-46ccae0f8d1d,a1c5bb38-4de7-4ca4-805a-7e3fc0419867,1fe99400-06e1-4964-956b-3861687e7da4,er deployment planned.\nEnterprise RAG platfor...,engineering
4,e001567a-7134-4290-9ebb-e7e07e069dcb,a1c5bb38-4de7-4ca4-805a-7e3fc0419867,1fe99400-06e1-4964-956b-3861687e7da4,ent | Status\nVector Search | Production\nHybr...,engineering


In [6]:
embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

In [7]:
sample_embedding = embedding_model.encode("hello world")

dimension = sample_embedding.shape[0]


print(dimension)

384


In [8]:
vector_indexes = {}

chunk_mappings = {}

In [9]:
def build_faiss_index(dataframe):

    texts = dataframe["content"].tolist()

    embeddings = embedding_model.encode(texts, convert_to_numpy=True)

    embeddings = embeddings.astype("float32")

    index = faiss.IndexFlatIP(embeddings.shape[1])

    faiss.normalize_L2(embeddings)

    index.add(embeddings)

    return index, dataframe

In [10]:
departments = children_df["department"].unique()


for department in departments:

    print("Building:", department)

    department_chunks = children_df[
        children_df["department"] == department
    ].reset_index(drop=True)

    index, mapping = build_faiss_index(department_chunks)

    vector_indexes[department] = index

    chunk_mappings[department] = mapping

Building: engineering
Building: finance
Building: hr


In [11]:
vector_indexes.keys()

dict_keys(['engineering', 'finance', 'hr'])

In [12]:
for name, index in vector_indexes.items():

    print(name, index.ntotal)

engineering 5
finance 5
hr 6


In [13]:
def search_department(query, department, top_k=3):

    query_embedding = embedding_model.encode([query], convert_to_numpy=True)

    query_embedding = query_embedding.astype("float32")

    faiss.normalize_L2(query_embedding)

    index = vector_indexes[department]

    scores, ids = index.search(query_embedding, top_k)

    results = []

    mapping = chunk_mappings[department]

    for score, idx in zip(scores[0], ids[0]):

        results.append(
            {
                "score": float(score),
                "content": mapping.iloc[idx]["content"],
                "parent_id": mapping.iloc[idx]["parent_id"],
            }
        )

    return results

In [14]:
search_department("employee healthcare benefits", "hr")

[{'score': 0.6041042804718018,
  'content': 'Employee Benefits 2026\nHealth Insurance\nDental Benefits\nRemote Work Policy\nHealth insurance coverage',
  'parent_id': 'fd7bb5e1-9ec6-470a-8726-bbdc04fcc6dc'},
 {'score': 0.562423586845398,
  'content': ' increased by 12%.\nThe company expanded hospitalization benefits.\nEmployees can now add dependents a',
  'parent_id': 'fd7bb5e1-9ec6-470a-8726-bbdc04fcc6dc'},
 {'score': 0.43751776218414307,
  'content': 'im internet reimbursement.\nWork-from-home equipment support continues.\nBenefit | Coverage\nHealth Ins',
  'parent_id': '4d127151-0af4-4b00-bdad-b66ad91aa656'}]

In [15]:
search_department("company revenue growth", "finance")

[{'score': 0.5142385959625244,
  'content': 'Annual Financial Report 2026\nRevenue Performance\nProfitability\nCloud Division\nRevenue reached $5.2 b',
  'parent_id': '9cb41e2d-b979-4c89-a091-01db42c3748e'},
 {'score': 0.46751123666763306,
  'content': 'oud division grew by 35%.\nAI products generated record profits.\nEnterprise customers expanded usage.',
  'parent_id': '7cdee370-fba0-4bc7-9fe1-a39230162073'},
 {'score': 0.4531693160533905,
  'content': 'illion.\nRevenue increased across all regions.\nSubscription services contributed strongly.\nOperating ',
  'parent_id': '9cb41e2d-b979-4c89-a091-01db42c3748e'}]

In [16]:
# =====================================================
# Load Semantic Router
# =====================================================

import pickle

from sklearn.metrics.pairwise import cosine_similarity

with open("artifacts/router_embeddings.pkl", "rb") as file:

    department_embeddings = pickle.load(file)


print(department_embeddings.keys())

dict_keys(['hr', 'finance', 'engineering'])


In [17]:
def route_query(query):

    query_embedding = embedding_model.encode(query)

    scores = {}

    for department, dept_embedding in department_embeddings.items():

        score = cosine_similarity(
            query_embedding.reshape(1, -1), dept_embedding.reshape(1, -1)
        )[0][0]

        scores[department] = float(score)

    best_department = max(scores, key=scores.get)

    return best_department, scores

In [19]:
# routed_search("What AI infrastructure projects are planned?")

In [20]:
department, scores = route_query("What AI infrastructure projects are planned?")


print(department)


scores

engineering


{'hr': 0.1434076428413391,
 'finance': 0.21328476071357727,
 'engineering': 0.5762346982955933}

In [21]:
os.makedirs("artifacts/faiss", exist_ok=True)


for department, index in vector_indexes.items():

    faiss.write_index(index, f"artifacts/faiss/{department}.index")

In [22]:
with open("artifacts/faiss/chunk_mapping.pkl", "wb") as file:

    pickle.dump(chunk_mappings, file)